# Challenge Sem Parar Empresas (Corpay) — Sprint 2
## Calculadora de Redução de Custos de Frota: Etapa de Inteligência e Diagnóstico
**FIAP · Graduação ON em Inteligência Artificial · 1TIAO**  
**Aluno:** Angelo Guilherme Sfreddo (RM: 566580)

---
### Objetivos Deste Notebook
1. **Ingestão e Normalização:** Unificar dados heterogêneos de custos (combustível em CSV, pedágio Sem Parar em JSON, manutenção em TXT posicional e multas/impostos em CSV);
2. **Auditoria de Qualidade (Cognitive Data Science):** Identificar e tratar valores negativos, campos nulos, registros duplicados, chaves órfãs e inconsistências físicas de consumo;
3. **Statistical Computing & Benchmark:** Segmentar a frota em grupos comparáveis (Leve Urbano, Médio Distribuição, Pesado Rodoviário) e calcular métricas de benchmark (média, mediana, quartis Q1/Q3);
4. **Análise de Gaps de Ineficiência:** Quantificar a diferença absoluta e percentual de cada veículo frente ao seu benchmark;
5. **Machine Learning & Diagnóstico:** Comparar a heurística por regras de negócio com modelos de classificação supervisionada (Árvore de Decisão, Random Forest, Regressão Logística);
6. **Ir Além:** Demonstração da Rede Neural Artificial MLP (MLPClassifier) e do extrator de faturas/comprovantes fiscais não estruturados.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
%matplotlib inline


## 1. Ingestão e Normalização de Fontes Heterogêneas
Carregando dados das fontes brutas e convertendo para o esquema padrão unificado.

In [ ]:
sys.path.insert(0, os.path.abspath('..'))
from src.ingestion.normalizador import executar_normalizacao

df_normalizado = executar_normalizacao()
df_normalizado.head(8)


## 2. Auditoria e Qualidade de Dados (Cognitive Data Science)
Executando os filtros de consistência e gerando a base integrada por veículo.

In [ ]:
from src.quality.auditoria_qualidade import executar_auditoria_qualidade

df_frota, df_inconsistencias = executar_auditoria_qualidade()
print(f'Total de inconsistências tratadas: {len(df_inconsistencias)}')
display(df_inconsistencias)


## 3. Benchmark Estatístico por Grupo Homogêneo e Gaps de Eficiência
Cálculo dos parâmetros centrais (Mediana, Q1, Q3, Média) e mensuração do Gap de Custo por KM.

In [ ]:
from src.benchmark.calculador_benchmark import calcular_benchmark_e_gaps

df_frota_bench, df_benchmarks = calcular_benchmark_e_gaps()
print('Tabela de Benchmarks Estatísticos por Grupo Homogêneo:')
display(df_benchmarks)


In [ ]:
ranking_cols = ['veiculo_id', 'modelo', 'perfil_operacao', 'custo_por_km', 'benchmark_custo_km', 'gap_percentual_custo_km', 'diagnostico_regras']
df_frota_bench.sort_values(by='gap_percentual_custo_km', ascending=False)[ranking_cols]


## 4. Machine Learning: Classificação Supervisionada de Eficiência
Treinando e comparando os modelos supervisionados e confrontando com as regras de negócio.

In [ ]:
from src.ml.classificador_eficiencia import treinar_e_avaliar_modelos

df_final, df_modelos_comp = treinar_e_avaliar_modelos()
print('Comparativo de Acurácia dos Modelos no Conjunto de Teste:')
display(df_modelos_comp)


## 5. Visualização Executiva dos Resultados
Exibindo os gráficos gerados pela suíte de analytics.

In [ ]:
from src.analytics.gerar_graficos_sprint2 import gerar_todos_os_graficos

gerar_todos_os_graficos()

from IPython.display import Image
Image(filename='../assets/diagramas/03_ranking_gaps_ineficiencia_frota.png')


## 6. Requisito 'Ir Além' (Demonstração de OCR / Parsing de Comprovantes)
Extraindo faturas fiscais não estruturadas e convertendo para o pipeline de dados.

In [ ]:
from src.ingestion.extrator_faturas import demonstrar_extracao

resultados_ocr = demonstrar_extracao()


## Conclusões da Sprint 2
1. **Pergunta 1 do Gestor ('Quanto a operação custa?'):** O TCO detalhado foi consolidado com precisão: o custo por km varia de R$ 0.709/km na frota leve até R$ 4.882/km na frota pesada, com distribuição percentual clara entre combustível, manutenção, pedágio Sem Parar e multas;
2. **Pergunta 2 do Gestor ('Onde está menos eficiente?'):** Foram identificados com exatidão os veículos críticos da operação (ex.: SP-V03 com +146.5% de desvio por avarias em suspensão e motor, SP-V14 com +96.5% por superaquecimento térmico registrado na telemetria ESP32 e SP-V22 com +39.2% na frota pesada);
3. **Preparação para a Sprint 3:** A base tratada, os benchmarks de referência e os modelos supervisionados fornecem o alicerce matemático e operacional para o Motor de Recomendações e Estimativa de Economia em Reais da Sprint 3.